In [ ]:
import os
import ee
import geemap
import ipywidgets as widgets

ee.Authenticate(force=True)


In [ ]:
roi = ee.FeatureCollection('projects/ee-zhuzhu94854693/assets/ronghe_shidi_cnn')

def ndpi(image):
    nd = image.expression(
        '(nir-(0.74*r+(1-0.74)*swir))/(nir+(0.74*r+(1-0.74)*swir))',
        {
            'nir': image.select("B5"),  
            'r': image.select("B4"),    
            'swir': image.select("B6")  
        }
    )
    return image.addBands(ee.Image(nd).rename("ndpi"))

def ndvi(image):
    nd = image.expression(
        '(nir-r)/(nir+r)',
        {
            'nir': image.select("B5"),  
            'r': image.select("B4")    
        }
    )
    return image.addBands(ee.Image(nd).rename("ndvi"))

def ndgi(image):
    nd = image.expression(
        '(0.65*g+0.35*nir-r)/(0.65*g+0.35*nir+r)',
        {
            'nir': image.select("B5"),  
            'r': image.select("B4"),    
            'g': image.select("B3")    
        }
    )
    return image.addBands(ee.Image(nd).rename("ndgi"))


def cloudMaskL8(image):
    qa = image.select('QA_PIXEL')
    
    cloud_bit = 1 << 3  
    shadow_bit = 1 << 4  
    snow_bit = 1 << 5  

    mask = qa.bitwiseAnd(cloud_bit).eq(0) \
          .And(qa.bitwiseAnd(shadow_bit).eq(0)) \
          .And(qa.bitwiseAnd(snow_bit).eq(0))
    
    return image.updateMask(mask)


datasat = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA")
    .filterBounds(roi)
    .filterDate('2020-01-01', '2020-12-31')
    .map(cloudMaskL8)
    .map(ndvi)
    .map(ndpi)
    .map(ndgi)
)

months = range(1, 13)
monthly_images = {}
monthly_images1 = {}


monthly_images = {}

monthly_data_1 = datasat.filter(ee.Filter.calendarRange(1, 3, "month"))
monthly_images[f'image_1'] = monthly_data_1.min().clip(roi)

monthly_data_2 = datasat.filter(ee.Filter.calendarRange(4, 6, "month"))
monthly_images[f'image_2'] = monthly_data_2.min().clip(roi)

monthly_data_3 = datasat.filter(ee.Filter.calendarRange(7, 9, "month"))
monthly_images[f'image_3'] = monthly_data_3.min().clip(roi)

monthly_data_4 = datasat.filter(ee.Filter.calendarRange(10, 12, "month"))
monthly_images[f'image_4'] = monthly_data_4.min().clip(roi)

bands_to_combine = []
bands_to_combine.append(monthly_images[f'image_1'].select("B1"))
bands_to_combine.append(monthly_images[f'image_1'].select("B2"))
bands_to_combine.append(monthly_images[f'image_1'].select("B3"))
bands_to_combine.append(monthly_images[f'image_1'].select("B4"))
bands_to_combine.append(monthly_images[f'image_1'].select("B5"))
bands_to_combine.append(monthly_images[f'image_1'].select("B6"))
bands_to_combine.append(monthly_images[f'image_1'].select("B7"))
bands_to_combine.append(monthly_images[f'image_1'].select("ndvi"))
bands_to_combine.append(monthly_images[f'image_1'].select("ndpi"))
bands_to_combine.append(monthly_images[f'image_1'].select("ndgi"))
bands_to_combine.append(monthly_images[f'image_2'].select("B1"))
bands_to_combine.append(monthly_images[f'image_2'].select("B2"))
bands_to_combine.append(monthly_images[f'image_2'].select("B3"))
bands_to_combine.append(monthly_images[f'image_2'].select("B4"))
bands_to_combine.append(monthly_images[f'image_2'].select("B5"))
bands_to_combine.append(monthly_images[f'image_2'].select("B6"))
bands_to_combine.append(monthly_images[f'image_2'].select("B7"))
bands_to_combine.append(monthly_images[f'image_2'].select("ndvi"))
bands_to_combine.append(monthly_images[f'image_2'].select("ndpi"))
bands_to_combine.append(monthly_images[f'image_2'].select("ndgi"))
bands_to_combine.append(monthly_images[f'image_3'].select("B1"))
bands_to_combine.append(monthly_images[f'image_3'].select("B2"))
bands_to_combine.append(monthly_images[f'image_3'].select("B3"))
bands_to_combine.append(monthly_images[f'image_3'].select("B4"))
bands_to_combine.append(monthly_images[f'image_3'].select("B5"))
bands_to_combine.append(monthly_images[f'image_3'].select("B6"))
bands_to_combine.append(monthly_images[f'image_3'].select("B7"))
bands_to_combine.append(monthly_images[f'image_3'].select("ndvi"))
bands_to_combine.append(monthly_images[f'image_3'].select("ndpi"))
bands_to_combine.append(monthly_images[f'image_3'].select("ndgi"))
bands_to_combine.append(monthly_images[f'image_4'].select("B1"))
bands_to_combine.append(monthly_images[f'image_4'].select("B2"))
bands_to_combine.append(monthly_images[f'image_4'].select("B3"))
bands_to_combine.append(monthly_images[f'image_4'].select("B4"))
bands_to_combine.append(monthly_images[f'image_4'].select("B5"))
bands_to_combine.append(monthly_images[f'image_4'].select("B6"))
bands_to_combine.append(monthly_images[f'image_4'].select("B7"))
bands_to_combine.append(monthly_images[f'image_4'].select("ndvi"))
bands_to_combine.append(monthly_images[f'image_4'].select("ndpi"))
bands_to_combine.append(monthly_images[f'image_4'].select("ndgi"))



datasat2 = ee.Image.cat(*bands_to_combine)
print(f"Total bands: {datasat2.bandNames().size().getInfo()}")
print("Band names:", datasat2.bandNames().getInfo())

points_train = ee.FeatureCollection('projects/ee-zhuzhu94854693/assets/train_3000_1960')
points_val = ee.FeatureCollection('projects/ee-zhuzhu94854693/assets/val_1000_280')
label = 'landcover'

training = datasat2.sampleRegions(**{
    'collection': points_train,
    'properties': [label],
    'scale': 30 
})
df = geemap.ee_to_pandas(training)
df.to_csv(r'C:\Users\aaf\Desktop\ge\shidi\samplepandas\samples_train_L8_40.csv', index=False)

val = datasat2.sampleRegions(**{
    'collection': points_val,
    'properties': [label],
    'scale': 30
})
df2 = geemap.ee_to_pandas(val)
df2.to_csv(r'C:\Users\aaf\Desktop\ge\shidi\samplepandas\samples_val_L8_40.csv', index=False)


In [ ]:

import numpy as np
import pandas as pd

import torch
import copy
from torch import nn
from torch import optim
from torch.nn import CrossEntropyLoss, BCEWithLogitsLoss
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset
import random
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.ensemble import RandomForestClassifier
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
from sklearn.utils.class_weight import compute_class_weight
import ast
from IPython.display import clear_output
import time

import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def dataloader_build(datasets_train, datasets_val, batch_size=100):
    train_dataset = datasets_train
    val_dataset = datasets_val
    train_datadloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_datadloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_datadloader, val_datadloader


class CNNDatasets(Dataset):

    def __init__(self, df: pd.DataFrame, normalize=True):
        super(CNNDatasets, self).__init__()
        df.reset_index(drop=True)
        self.band_df = df[[
                          'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'ndvi', 'ndpi', 'ndgi', 
                          'B1_1', 'B2_1', 'B3_1', 'B4_1', 'B5_1', 'B6_1', 'B7_1', 'ndvi_1', 'ndpi_1', 'ndgi_1', 
                          'B1_2', 'B2_2', 'B3_2', 'B4_2', 'B5_2', 'B6_2', 'B7_2', 'ndvi_2', 'ndpi_2', 'ndgi_2', 
                          'B1_3', 'B2_3', 'B3_3', 'B4_3', 'B5_3', 'B6_3', 'B7_3', 'ndvi_3', 'ndpi_3', 'ndgi_3',
                          
           
        ]].copy()

        self.label_df = df[['landcover']].copy()


    def __len__(self):
        return self.band_df.shape[0]

    def __getitem__(self, i):
        bands = np.array(self.band_df.iloc[i])
        label = self.label_df.iloc[i]['landcover']
        return bands, label





class CNN(nn.Module):
    def __init__(self,in_channels, out_classes):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels, 128, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(128)

        self.conv6 = nn.Conv1d(128, 64, kernel_size=1)

        self.conv7 = nn.Conv1d(64, out_classes, kernel_size=1)


        self.dropout = nn.Dropout(p=0.5)
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.1)

    def forward(self, x):

        x = x.unsqueeze(2)
        x = self.leaky_relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)

        x = self.conv6(x)

        x = self.conv7(x)
        x = x.squeeze(2)

        return x

    def predict(self, x):
        return F.softmax(self.forward(x), dim=1)




if __name__ == '__main__':
    set_seed(42)
   
    df = pd.read_csv(r'C:\Users\aaf\Desktop\ge\shidi\samplepandas\samples_train_L8_40.csv')
    df2 = pd.read_csv(r'C:\Users\aaf\Desktop\ge\shidi\samplepandas\samples_val_L8_40.csv')


    dataset_train = CNNDatasets(df)
    dataset_val = CNNDatasets(df2)
    train_datadloader, val_datadloader = dataloader_build(dataset_train, dataset_val)


    model = CNN(40, 13).float().to(device)
    lossfunc = CrossEntropyLoss().to(device)


    print('kaishi')

    max_epoch = 10000
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    lr_scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=150, T_mult=2)





    best_accuracy = 0.0  

    for epoch in range(max_epoch):
        # train
        model.train()
        correct_train, total_train = 0, 0
        for bands, label in train_datadloader:

            if bands.shape[0] == 1:
                continue
                
            bands, label = bands.float().to(device), label.to(device)
            optimizer.zero_grad()
            pred = model(bands)
            _, predicted = torch.max(pred, 1)  
            correct_train += (predicted == label).sum().item()  
            total_train += label.size(0)  

            one_hot_labels = torch.nn.functional.one_hot(label, num_classes=13).float()
            pred_probs = F.softmax(pred, dim=1)
            pred_log_probs = torch.log(pred_probs)
            loss = lossfunc(pred, label.long())

            loss.backward()
            optimizer.step()

   
        train_accuracy = 100 * correct_train / total_train
        all_preds = []
        all_label = []
       
        model.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for bands, label in val_datadloader:
                bands, label = bands.float().to(device), label.to(device)
                pred = model(bands)
                _, predicted = torch.max(pred, 1)
                correct_val += (predicted == label).sum().item()
                total_val += label.size(0)
                pred_labels = torch.argmax(pred, dim=1).cpu().numpy()

        
        val_accuracy = 100 * correct_val / total_val

      
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            
            torch.save(model.state_dict(), r'C:\Users\aaf\Desktop\ge\shidi\pth\best_model_pytorch_120_L8_40_3.pth')


       
        clear_output(wait=True)
        print(f'Epoch [{epoch + 1}/{max_epoch}]\n'
              f'Train Accuracy: {train_accuracy:.2f}%\n'
              f'Val Accuracy: {val_accuracy:.2f}%\n'
              f'Best Val Accuracy: {best_accuracy:.2f}%')
        time.sleep(0.1)  

        lr_scheduler.step(val_accuracy)





In [ ]:
import ee
import torch
from torch import nn
import geemap
Map=geemap.Map()
import random





class CNN(nn.Module):
    def __init__(self,in_channels, out_classes):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels, 128, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(128)

        self.conv6 = nn.Conv1d(128, 64, kernel_size=1)

        self.conv7 = nn.Conv1d(64, out_classes, kernel_size=1)


        self.dropout = nn.Dropout(p=0.5)
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.1)

    def forward(self, x):

        x = x.unsqueeze(2)
        x = self.leaky_relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)

        x = self.conv6(x)

        x = self.conv7(x)
        x = x.squeeze(2)

        return x

    def predict(self, x):
        return F.softmax(self.forward(x), dim=1)




model_path = "C:/Users/aaf/Desktop/ge/shidi/pth/best_model_pytorch_120_L8_40_3.pth"
model = CNN(in_channels=40, out_classes=13)
model.load_state_dict(torch.load(model_path))
model.eval()

def fuse_conv_bn(conv, bn):
    eps = bn.eps
    std = (bn.running_var + eps).sqrt()

    w = conv.weight.detach()  # shape: (out_ch, in_ch, 1)
    b = conv.bias.detach() if conv.bias is not None else torch.zeros_like(bn.running_mean)

    w_fused = w * (bn.weight / std).reshape(-1, 1, 1)
    b_fused = (b - bn.running_mean) / std * bn.weight + bn.bias

    return w_fused.detach().numpy(), b_fused.detach().numpy()

w1, b1 = fuse_conv_bn(model.conv1, model.bn1)
w6 = model.conv6.weight.detach().numpy()
b6 = model.conv6.bias.detach().numpy()
w7 = model.conv7.weight.detach().numpy()
b7 = model.conv7.bias.detach().numpy()


def leaky_relu(image, alpha=0.1):
    return image.where(image.lt(0), image.multiply(alpha))
    
def conv1d_1x1(image, weights_tensor, bias_tensor, prefix="conv"):
    """
    Performs 1x1 Conv1D over ee.Image bands using provided weights and biases.
    
    Args:
        image (ee.Image): Input image with multiple bands.
        weights_tensor (numpy.ndarray): Shape (out_channels, in_channels, 1)
        bias_tensor (numpy.ndarray): Shape (out_channels,)
        prefix (str): Output band prefix for naming clarity.
        
    Returns:
        ee.Image: Output image after 1x1 convolution.
    """
    out_channels, in_channels, kernel_size = weights_tensor.shape
    assert kernel_size == 1, "Only kernel_size=1 is supported."

    output_bands = []
    input_bands = image.bandNames().getInfo()

    for out_ch in range(out_channels):
        sum_img = None
        for in_ch in range(in_channels):
            band_name = input_bands[in_ch]
            band = image.select(band_name)
            w = weights_tensor[out_ch, in_ch, 0]

            weighted_band = band.multiply(float(w))
            sum_img = weighted_band if sum_img is None else sum_img.add(weighted_band)

        sum_img = sum_img.add(float(bias_tensor[out_ch]))
        sum_img = sum_img.rename(f'{prefix}_{out_ch}')  
        output_bands.append(sum_img)

    return ee.Image.cat(output_bands)

bands = [

          'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'ndvi', 'ndpi', 'ndgi', 
          'B1_1', 'B2_1', 'B3_1', 'B4_1', 'B5_1', 'B6_1', 'B7_1', 'ndvi_1', 'ndpi_1', 'ndgi_1', 
          'B1_2', 'B2_2', 'B3_2', 'B4_2', 'B5_2', 'B6_2', 'B7_2', 'ndvi_2', 'ndpi_2', 'ndgi_2', 
          'B1_3', 'B2_3', 'B3_3', 'B4_3', 'B5_3', 'B6_3', 'B7_3', 'ndvi_3', 'ndpi_3', 'ndgi_3',
    ]


def leaky_relu(image, alpha=0.1):
    return image.where(image.lt(0), image.multiply(alpha))


def sigmoid(image):
    return image.multiply(-1).exp().add(1).pow(-1)


def relu(image):
    return image.max(0)




def softmax(image):
    array = image.toArray()
    exp = array.exp()
    sum_exp = exp.arrayReduce(ee.Reducer.sum(), [0])
    probs = exp.divide(sum_exp)
    return probs.arrayProject([0]).arrayFlatten([['class_0', 'class_1', ..., 'class_n']])


def pixel_based_ETACNN(image):
    x0 = image.select(bands)  # 按固定顺序选择

    # === conv1 + bn1 fused ===
    x = conv1d_1x1(x0, w1, b1)
    x1 = leaky_relu(x, 0.1)

    # === conv6 ===
    x = conv1d_1x1(x1, w6, b6)
    x2 = leaky_relu(x, 0.1)

    # === conv7 ===
    x3 = conv1d_1x1(x2, w7, b7)

    return x0, x1, x2, x3 



roi = ee.FeatureCollection('projects/ee-zhuzhu94854693/assets/ronghe_shidi_cnn') 



def ndpi(image):
    nd = image.expression(
        '(nir-(0.74*r+(1-0.74)*swir))/(nir+(0.74*r+(1-0.74)*swir))',
        {
            'nir': image.select("B5"),  
            'r': image.select("B4"),    
            'swir': image.select("B6")  
        }
    )
    return image.addBands(ee.Image(nd).rename("ndpi"))

def ndvi(image):
    nd = image.expression(
        '(nir-r)/(nir+r)',
        {
            'nir': image.select("B5"),  
            'r': image.select("B4")     
        }
    )
    return image.addBands(ee.Image(nd).rename("ndvi"))

def ndgi(image):
    nd = image.expression(
        '(0.65*g+0.35*nir-r)/(0.65*g+0.35*nir+r)',
        {
            'nir': image.select("B5"),  
            'r': image.select("B4"),    
            'g': image.select("B3")     
        }
    )
    return image.addBands(ee.Image(nd).rename("ndgi"))

def cloudMaskL8(image):
    qa = image.select('QA_PIXEL')
    
    cloud_bit = 1 << 3  
    shadow_bit = 1 << 4 
    snow_bit = 1 << 5 
   
    mask = qa.bitwiseAnd(cloud_bit).eq(0) \
          .And(qa.bitwiseAnd(shadow_bit).eq(0)) \
          .And(qa.bitwiseAnd(snow_bit).eq(0))
    
    return image.updateMask(mask)


datasat = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA")
    .filterBounds(roi)
    .filterDate('2020-01-01', '2020-12-31')
    .map(cloudMaskL8)
    .map(ndvi)
    .map(ndpi)
    .map(ndgi)
)


months = range(1, 13)
monthly_images = {}
monthly_images1 = {}



monthly_images = {}

monthly_data_1 = datasat.filter(ee.Filter.calendarRange(1, 3, "month"))
monthly_images[f'image_1'] = monthly_data_1.min().clip(roi)

monthly_data_2 = datasat.filter(ee.Filter.calendarRange(4, 6, "month"))
monthly_images[f'image_2'] = monthly_data_2.min().clip(roi)

monthly_data_3 = datasat.filter(ee.Filter.calendarRange(7, 9, "month"))
monthly_images[f'image_3'] = monthly_data_3.min().clip(roi)

monthly_data_4 = datasat.filter(ee.Filter.calendarRange(10, 12, "month"))
monthly_images[f'image_4'] = monthly_data_4.min().clip(roi)

bands_to_combine = []
bands_to_combine.append(monthly_images[f'image_1'].select("B1"))
bands_to_combine.append(monthly_images[f'image_1'].select("B2"))
bands_to_combine.append(monthly_images[f'image_1'].select("B3"))
bands_to_combine.append(monthly_images[f'image_1'].select("B4"))
bands_to_combine.append(monthly_images[f'image_1'].select("B5"))
bands_to_combine.append(monthly_images[f'image_1'].select("B6"))
bands_to_combine.append(monthly_images[f'image_1'].select("B7"))
bands_to_combine.append(monthly_images[f'image_1'].select("ndvi"))
bands_to_combine.append(monthly_images[f'image_1'].select("ndpi"))
bands_to_combine.append(monthly_images[f'image_1'].select("ndgi"))
bands_to_combine.append(monthly_images[f'image_2'].select("B1"))
bands_to_combine.append(monthly_images[f'image_2'].select("B2"))
bands_to_combine.append(monthly_images[f'image_2'].select("B3"))
bands_to_combine.append(monthly_images[f'image_2'].select("B4"))
bands_to_combine.append(monthly_images[f'image_2'].select("B5"))
bands_to_combine.append(monthly_images[f'image_2'].select("B6"))
bands_to_combine.append(monthly_images[f'image_2'].select("B7"))
bands_to_combine.append(monthly_images[f'image_2'].select("ndvi"))
bands_to_combine.append(monthly_images[f'image_2'].select("ndpi"))
bands_to_combine.append(monthly_images[f'image_2'].select("ndgi"))
bands_to_combine.append(monthly_images[f'image_3'].select("B1"))
bands_to_combine.append(monthly_images[f'image_3'].select("B2"))
bands_to_combine.append(monthly_images[f'image_3'].select("B3"))
bands_to_combine.append(monthly_images[f'image_3'].select("B4"))
bands_to_combine.append(monthly_images[f'image_3'].select("B5"))
bands_to_combine.append(monthly_images[f'image_3'].select("B6"))
bands_to_combine.append(monthly_images[f'image_3'].select("B7"))
bands_to_combine.append(monthly_images[f'image_3'].select("ndvi"))
bands_to_combine.append(monthly_images[f'image_3'].select("ndpi"))
bands_to_combine.append(monthly_images[f'image_3'].select("ndgi"))
bands_to_combine.append(monthly_images[f'image_4'].select("B1"))
bands_to_combine.append(monthly_images[f'image_4'].select("B2"))
bands_to_combine.append(monthly_images[f'image_4'].select("B3"))
bands_to_combine.append(monthly_images[f'image_4'].select("B4"))
bands_to_combine.append(monthly_images[f'image_4'].select("B5"))
bands_to_combine.append(monthly_images[f'image_4'].select("B6"))
bands_to_combine.append(monthly_images[f'image_4'].select("B7"))
bands_to_combine.append(monthly_images[f'image_4'].select("ndvi"))
bands_to_combine.append(monthly_images[f'image_4'].select("ndpi"))
bands_to_combine.append(monthly_images[f'image_4'].select("ndgi"))


datasat2 = ee.Image.cat(*bands_to_combine)


image = datasat2.clip(roi)


x0, x1, x2, x3 = pixel_based_ETACNN(image)




roi2222 = ee.FeatureCollection('projects/ee-zhuzhu94854693/assets/ronghe_shidi_cnn') 
train_point = ee.FeatureCollection('projects/ee-zhuzhu94854693/assets/train_3000_1960')


x0_bands = [
          'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'ndvi', 'ndpi', 'ndgi', 
          'B1_1', 'B2_1', 'B3_1', 'B4_1', 'B5_1', 'B6_1', 'B7_1', 'ndvi_1', 'ndpi_1', 'ndgi_1', 
          'B1_2', 'B2_2', 'B3_2', 'B4_2', 'B5_2', 'B6_2', 'B7_2', 'ndvi_2', 'ndpi_2', 'ndgi_2', 
          'B1_3', 'B2_3', 'B3_3', 'B4_3', 'B5_3', 'B6_3', 'B7_3', 'ndvi_3', 'ndpi_3', 'ndgi_3',
] 
training = x0.select(x0_bands).sampleRegions(
    collection=train_point,
    properties=['landcover'],
    scale=30,
    tileScale=16
)


rf_classifier = ee.Classifier.smileRandomForest(numberOfTrees=800).setOutputMode('MULTIPROBABILITY').train(
        features=training,
        classProperty='landcover',
        inputProperties=x0_bands
    )

rf_classified_x0 = x0.select(x0_bands).classify(rf_classifier).clip(roi2222)

def extract_prob(i):
    band_index = ee.Number(i)
    band_name = 'conv_' + str(band_index.getInfo()) 
    return rf_array_x0.arrayGet(band_index).rename(band_name)

rf_array_x0 = rf_classified_x0.toArray()


prob_rf_classified_x0 = [extract_prob(ee.Number(i)) for i in range(13)]
prob_rf_classified_x0 = ee.Image.cat(prob_rf_classified_x0)




x1_bands = [
            'conv_0', 'conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5', 'conv_6', 'conv_7', 'conv_8', 'conv_9', 
            'conv_10', 'conv_11', 'conv_12', 'conv_13', 'conv_14', 'conv_15', 'conv_16', 'conv_17', 'conv_18', 'conv_19', 
            'conv_20', 'conv_21', 'conv_22', 'conv_23', 'conv_24', 'conv_25', 'conv_26', 'conv_27', 'conv_28', 'conv_29', 
            'conv_30', 'conv_31', 'conv_32', 'conv_33', 'conv_34', 'conv_35', 'conv_36', 'conv_37', 'conv_38', 'conv_39', 
            'conv_40', 'conv_41', 'conv_42', 'conv_43', 'conv_44', 'conv_45', 'conv_46', 'conv_47', 'conv_48', 'conv_49', 
            'conv_50', 'conv_51', 'conv_52', 'conv_53', 'conv_54', 'conv_55', 'conv_56', 'conv_57', 'conv_58', 'conv_59', 
            'conv_60', 'conv_61', 'conv_62', 'conv_63', 'conv_64', 'conv_65', 'conv_66', 'conv_67', 'conv_68', 'conv_69', 
            'conv_70', 'conv_71', 'conv_72', 'conv_73', 'conv_74', 'conv_75', 'conv_76', 'conv_77', 'conv_78', 'conv_79', 
            'conv_80', 'conv_81', 'conv_82', 'conv_83', 'conv_84', 'conv_85', 'conv_86', 'conv_87', 'conv_88', 'conv_89', 
            'conv_90', 'conv_91', 'conv_92', 'conv_93', 'conv_94', 'conv_95', 'conv_96', 'conv_97', 'conv_98', 'conv_99', 
            'conv_100', 'conv_101', 'conv_102', 'conv_103', 'conv_104', 'conv_105', 'conv_106', 'conv_107', 'conv_108', 'conv_109',
            'conv_110', 'conv_111', 'conv_112', 'conv_113', 'conv_114', 'conv_115', 'conv_116', 'conv_117', 'conv_118', 'conv_119',
            'conv_120', 'conv_121', 'conv_122', 'conv_123', 'conv_124', 'conv_125', 'conv_126', 'conv_127', 
          
] 

training = x1.select(x1_bands).sampleRegions(
    collection=train_point,
    properties=['landcover'],
    scale=30,
    tileScale=16
)
 
rf_classifier = ee.Classifier.smileRandomForest(numberOfTrees=800).setOutputMode('MULTIPROBABILITY').train(
        features=training,
        classProperty='landcover',
        inputProperties=x1_bands
    )

rf_classified_x1 = x1.select(x1_bands).classify(rf_classifier).clip(roi2222)

def extract_prob(i):
    band_index = ee.Number(i)
    band_name = 'conv_' + str(band_index.getInfo()) 
    return rf_array_x1.arrayGet(band_index).rename(band_name)

rf_array_x1 = rf_classified_x1.toArray()

prob_rf_classified_x1 = [extract_prob(ee.Number(i)) for i in range(13)]
prob_rf_classified_x1 = ee.Image.cat(prob_rf_classified_x1)


final = prob_rf_classified_x0.multiply(prob_rf_classified_x1).multiply(x3) 
final = final.toArray().arrayArgmax()
final = final.arrayFlatten([['class']]) 

Map.addLayer(final)
Map.addLayer(prob_rf_classified_x0)
Map.addLayer(prob_rf_classified_x1)
Map.addLayer(x3)
Map


task3 = ee.batch.Export.image.toDrive(
    image=final.clip(roi2222), 
    description='final',
    folder='GEE_exports',  
    fileNamePrefix='final',
    region=roi2222.geometry(),
    scale=30, 
    maxPixels=1e13
)

task3.start()

print("The export task has been submitted. Please confirm and download it from the GEE Task panel.")
